# Notebook 09: Publication-Quality Visualisations

**Author:** Anthony Amit Biswas

## What this notebook does

Creates consistent research figures and summary assets from the frozen, validated error-analysis outputs without recomputing predictions or metrics.


## Step 0 — Environment and Google Drive

The notebook mounts Google Drive in Colab and imports the libraries required for final figure production.

In [ ]:
from pathlib import Path
import json
import math
import re
import shutil
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNNING_IN_COLAB = True
except Exception:
    RUNNING_IN_COLAB = False

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version : {np.__version__}")

## Step 1 — Configuration

Normally, only `PROJECT_ROOT` needs to be changed.

Notebook 08 is expected at:

`/content/drive/MyDrive/Dissertation/outputs/notebook_08_error_analysis`

In [ ]:
# USER CONFIGURATION

PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

NOTEBOOK_08_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "notebook_08_error_analysis"
)

NOTEBOOK_08_TABLE_DIRECTORY = (
    NOTEBOOK_08_DIRECTORY / "tables"
)

NOTEBOOK_08_REPORT_DIRECTORY = (
    NOTEBOOK_08_DIRECTORY / "reports"
)

OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "notebook_09_publication_assets"
)

DISSERTATION_DIRECTORY = (
    OUTPUT_DIRECTORY / "dissertation_figures"
)

POSTER_DIRECTORY = (
    OUTPUT_DIRECTORY / "poster_figures"
)

PRESENTATION_DIRECTORY = (
    OUTPUT_DIRECTORY / "presentation_figures"
)

SOCIAL_DIRECTORY = (
    OUTPUT_DIRECTORY / "social_media_figures"
)

TABLE_DIRECTORY = (
    OUTPUT_DIRECTORY / "tables"
)

REPORT_DIRECTORY = (
    OUTPUT_DIRECTORY / "reports"
)

for directory in [
    OUTPUT_DIRECTORY,
    DISSERTATION_DIRECTORY,
    POSTER_DIRECTORY,
    PRESENTATION_DIRECTORY,
    SOCIAL_DIRECTORY,
    TABLE_DIRECTORY,
    REPORT_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Notebook 08 input directory:")
print(NOTEBOOK_08_DIRECTORY)

print("\nNotebook 09 output directory:")
print(OUTPUT_DIRECTORY)

## Step 2 — Input registry

The notebook reads only Notebook 08 tables and reports. The registry deliberately excludes obsolete filenames from earlier Notebook 08 versions.

In [ ]:
INPUT_FILES = {
    "corpus_summary":
        "table_01_corpus_summary.csv",

    "overall_performance":
        "table_02_overall_performance.csv",

    "fp_by_category":
        "table_03_fp_by_category.csv",

    "fn_by_category":
        "table_04_fn_by_category.csv",

    "fn_root_causes":
        "table_04_false_negative_root_causes.csv",

    "boundary_summary":
        "table_05_boundary_summary.csv",

    "category_summary":
        "table_06_category_classification_summary.csv",

    "assertion_summary":
        "table_07_assertion_research_summary.csv",

    "assertion_error_pairs":
        "table_07_assertion_error_pairs.csv",

    "temporality_summary":
        "table_08_temporality_research_summary.csv",

    "temporality_error_pairs":
        "table_08_temporality_error_pairs.csv",

    "category_metrics":
        "table_09_per_category_exact_span_category_metrics.csv",

    "category_support":
        "table_09_category_support_vs_f1.csv",

    "primary_error_taxonomy":
        "table_10_primary_error_taxonomy.csv",

    "observed_errors":
        "table_11_observed_errors_by_dimension_non_additive.csv",

    "interpretation_summary":
        "table_13_interpretation_summary.csv",

    "poster_results":
        "table_14_poster_results_shortlist.csv",

    "poster_manifest":
        "table_15_poster_figure_manifest.csv",
}

REPORT_FILES = {
    "notebook_08_validation":
        "notebook_08_final_validation.csv",

    "notebook_08_report":
        "notebook_08_error_analysis_report.json",
}

## Step 3 — Visual identity and output standards

The figures use one consistent visual system:

- dark navy for principal results;
- teal for validated high performance;
- amber for caution or intermediate performance;
- red for errors or limitations;
- neutral grey for contextual information.

All figures are exported as:

- high-resolution PNG;
- vector PDF;
- vector SVG.

No raw patient text is placed in any public-facing figure.

In [ ]:
# PROJECT VISUAL IDENTITY

NAVY = "#16324F"
BLUE = "#245C8A"
TEAL = "#2A9D8F"
AMBER = "#E9A23B"
RED = "#C44536"
PURPLE = "#7251A3"
GREY = "#6B7280"
LIGHT_GREY = "#E5E7EB"
PALE_GREY = "#F5F7FA"
WHITE = "#FFFFFF"
BLACK = "#111827"

CATEGORY_COLOURS = {
    "Exact span": BLUE,
    "Exact span + category": TEAL,
    "Relaxed overlap + category": PURPLE,
    "Strict full extraction": AMBER,
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#B8C0CC",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": WHITE,
    "axes.facecolor": WHITE,
    "savefig.facecolor": WHITE,
    "legend.frameon": False,
})

print("Publication visual style configured.")

## Step 4 — Utility functions

In [ ]:
def load_table(key, required=True):
    path = (
        NOTEBOOK_08_TABLE_DIRECTORY
        / INPUT_FILES[key]
    )

    if not path.exists():
        if required:
            raise FileNotFoundError(path)

        print(f"[SKIP] Optional table not found: {path.name}")
        return pd.DataFrame()

    dataframe = pd.read_csv(path)

    print(
        f"[LOAD] {path.name}: "
        f"{len(dataframe):,} rows × "
        f"{len(dataframe.columns):,} columns"
    )

    return dataframe


def first_existing_column(
    dataframe,
    candidates,
    required=False,
):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    if required:
        raise KeyError(
            "None of the expected columns were found: "
            + ", ".join(candidates)
        )

    return None


def clean_label(value):
    return (
        str(value)
        .strip()
        .replace("_", " ")
        .title()
    )


def save_publication_figure(
    figure,
    filename_stem,
    destinations,
    dpi=450,
):
    for destination in destinations:
        destination.mkdir(
            parents=True,
            exist_ok=True,
        )

        for extension in ["png", "pdf", "svg"]:
            path = (
                destination
                / f"{filename_stem}.{extension}"
            )

            figure.savefig(
                path,
                dpi=dpi,
                bbox_inches="tight",
                pad_inches=0.12,
            )

    print(
        f"[SAVE] {filename_stem} "
        f"→ {len(destinations)} destination(s)"
    )


def save_table(dataframe, filename):
    path = TABLE_DIRECTORY / filename
    dataframe.to_csv(path, index=False)
    print(f"[SAVE] {path.name}")
    return path


def add_bar_labels(
    axis,
    decimals=2,
    suffix="",
    padding=3,
):
    for container in axis.containers:
        labels = []

        for bar in container:
            value = bar.get_height()

            if not np.isfinite(value):
                labels.append("")
            else:
                labels.append(
                    f"{value:.{decimals}f}{suffix}"
                )

        axis.bar_label(
            container,
            labels=labels,
            padding=padding,
            fontsize=9,
        )


def metric_lookup(
    dataframe,
    evaluation_column="Evaluation",
):
    output = {}

    for _, row in dataframe.iterrows():
        output[str(row[evaluation_column])] = row.to_dict()

    return output


def normalise_confusion_from_pairs(
    pair_dataframe,
    gold_column,
    predicted_column,
    count_column,
):
    matrix = pd.pivot_table(
        pair_dataframe,
        index=gold_column,
        columns=predicted_column,
        values=count_column,
        aggfunc="sum",
        fill_value=0,
    )

    row_totals = matrix.sum(axis=1).replace(0, np.nan)

    return matrix.div(row_totals, axis=0).fillna(0)


def plot_heatmap(
    matrix,
    title,
    filename_stem,
    destinations,
    value_format=".0%",
):
    figure_width = max(6.5, 1.15 * len(matrix.columns))
    figure_height = max(4.8, 0.85 * len(matrix.index) + 2)

    figure, axis = plt.subplots(
        figsize=(figure_width, figure_height)
    )

    cmap = LinearSegmentedColormap.from_list(
        "project_heatmap",
        [WHITE, "#D9EEF2", TEAL, NAVY],
    )

    image = axis.imshow(
        matrix.values,
        cmap=cmap,
        vmin=0,
        vmax=max(1.0, float(matrix.values.max())),
        aspect="auto",
    )

    axis.set_xticks(range(len(matrix.columns)))
    axis.set_xticklabels(
        [clean_label(x) for x in matrix.columns],
        rotation=30,
        ha="right",
    )

    axis.set_yticks(range(len(matrix.index)))
    axis.set_yticklabels(
        [clean_label(x) for x in matrix.index]
    )

    axis.set_xlabel("Predicted label")
    axis.set_ylabel("Gold-standard label")
    axis.set_title(title, pad=14)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = float(
                matrix.iloc[row_index, column_index]
            )

            label = format(value, value_format)

            text_colour = (
                WHITE
                if value >= 0.55
                else BLACK
            )

            axis.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                fontsize=9,
                color=text_colour,
                fontweight=(
                    "bold"
                    if value >= 0.50
                    else "normal"
                ),
            )

    colourbar = figure.colorbar(
        image,
        ax=axis,
        fraction=0.035,
        pad=0.03,
    )
    colourbar.set_label("Row proportion")

    figure.tight_layout()

    save_publication_figure(
        figure,
        filename_stem,
        destinations,
    )

    plt.show()
    plt.close(figure)

## Step 5 — Validate Notebook 08 before visualisation

Notebook 09 must stop if Notebook 08 did not pass its final checks.

In [ ]:
validation_path = (
    NOTEBOOK_08_REPORT_DIRECTORY
    / REPORT_FILES["notebook_08_validation"]
)

if not validation_path.exists():
    raise FileNotFoundError(
        "Notebook 08 validation report was not found: "
        + str(validation_path)
    )

notebook_08_validation_df = pd.read_csv(
    validation_path
)

display(notebook_08_validation_df)

passed_column = first_existing_column(
    notebook_08_validation_df,
    ["Passed", "passed"],
    required=True,
)

failed_validation_df = (
    notebook_08_validation_df.loc[
        ~notebook_08_validation_df[
            passed_column
        ].astype(bool)
    ]
)

if not failed_validation_df.empty:
    display(failed_validation_df)

    raise RuntimeError(
        "Notebook 08 contains failed validation checks. "
        "Notebook 09 has stopped to protect the frozen results."
    )

print("Notebook 08 validation passed.")

## Step 6 — Load frozen Notebook 08 tables

In [ ]:
data = {
    key: load_table(key)
    for key in INPUT_FILES
}

print(
    "\nLoaded "
    f"{len(data):,} frozen Notebook 08 tables."
)

# Part A — Headline performance

## Step 7 — Publication Figure 1: Evaluation performance ladder

This is the principal Results figure. It shows how performance changes as the evaluation becomes more demanding.

In [ ]:
performance_df = data["overall_performance"].copy()

evaluation_column = first_existing_column(
    performance_df,
    ["Evaluation", "evaluation"],
    required=True,
)

precision_column = first_existing_column(
    performance_df,
    ["Precision", "precision"],
    required=True,
)

recall_column = first_existing_column(
    performance_df,
    ["Recall", "recall"],
    required=True,
)

f1_column = first_existing_column(
    performance_df,
    ["F1", "f1", "F1-score"],
    required=True,
)

preferred_order = [
    "Exact span",
    "Exact span + category",
    "Relaxed overlap + category",
    "Strict full extraction",
]

performance_df[evaluation_column] = (
    performance_df[evaluation_column]
    .astype(str)
)

performance_df["_order"] = (
    performance_df[evaluation_column]
    .map({
        name: index
        for index, name
        in enumerate(preferred_order)
    })
)

performance_df = (
    performance_df
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

figure, axis = plt.subplots(figsize=(11, 6.4))

x_positions = np.arange(len(performance_df))
bar_width = 0.23

metric_specs = [
    ("Precision", precision_column, BLUE),
    ("Recall", recall_column, TEAL),
    ("F1-score", f1_column, AMBER),
]

for offset_index, (
    metric_label,
    column,
    colour,
) in enumerate(metric_specs):
    offset = (
        offset_index - 1
    ) * bar_width

    bars = axis.bar(
        x_positions + offset,
        performance_df[column],
        width=bar_width,
        label=metric_label,
        color=colour,
    )

    labels = [
        f"{value:.3f}"
        for value in performance_df[column]
    ]

    axis.bar_label(
        bars,
        labels=labels,
        padding=3,
        fontsize=8.5,
    )

axis.set_xticks(x_positions)
axis.set_xticklabels(
    performance_df[evaluation_column],
    rotation=10,
    ha="center",
)

axis.set_ylim(0, 1.06)
axis.set_ylabel("Score")
axis.set_title(
    "Clinical complication extraction performance "
    "under increasingly strict evaluation"
)
axis.legend(
    ncol=3,
    loc="upper right",
)

axis.grid(
    axis="y",
    alpha=0.20,
    linewidth=0.7,
)

figure.text(
    0.01,
    0.01,
    "Exact-span and relaxed evaluations assess mention/category detection; "
    "strict full extraction additionally requires correct assertion and temporality.",
    fontsize=8.5,
    color=GREY,
)

figure.tight_layout(rect=[0, 0.045, 1, 1])

save_publication_figure(
    figure,
    "figure_01_evaluation_performance_ladder",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
        SOCIAL_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

## Step 8 — Publication Figure 2: F1 degradation across the evaluation pipeline

This figure isolates the F1 effect of adding category and contextual requirements.

In [ ]:
performance_map = metric_lookup(
    performance_df,
    evaluation_column=evaluation_column,
)

pipeline_names = [
    "Exact span",
    "Exact span + category",
    "Strict full extraction",
]

pipeline_values = [
    float(performance_map[name][f1_column])
    for name in pipeline_names
]

figure, axis = plt.subplots(figsize=(9.5, 5.8))

x_positions = np.arange(len(pipeline_names))

bars = axis.bar(
    x_positions,
    pipeline_values,
    color=[BLUE, TEAL, AMBER],
    width=0.62,
)

axis.set_xticks(x_positions)
axis.set_xticklabels(
    [
        "Span detection",
        "Span + category",
        "Full extraction",
    ]
)

axis.set_ylim(0, 1.05)
axis.set_ylabel("F1-score")
axis.set_title(
    "Performance decreases when contextual attributes "
    "are required"
)

axis.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in pipeline_values
    ],
    padding=4,
    fontsize=11,
    fontweight="bold",
)

for index in range(len(pipeline_values) - 1):
    difference = (
        pipeline_values[index + 1]
        - pipeline_values[index]
    )

    midpoint = (
        x_positions[index]
        + x_positions[index + 1]
    ) / 2

    axis.annotate(
        f"{difference:+.3f}",
        xy=(
            midpoint,
            min(
                pipeline_values[index],
                pipeline_values[index + 1],
            ) + 0.055,
        ),
        ha="center",
        va="bottom",
        fontsize=10,
        color=RED,
        fontweight="bold",
    )

axis.grid(
    axis="y",
    alpha=0.20,
)

figure.text(
    0.01,
    0.01,
    "The largest reduction occurs when assertion and temporality "
    "must also be correct.",
    fontsize=9,
    color=GREY,
)

figure.tight_layout(rect=[0, 0.05, 1, 1])

save_publication_figure(
    figure,
    "figure_02_contextual_attribute_penalty",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

## Step 9 — Poster metric-card panel

A compact, poster-readable summary replaces the less informative corpus-count bar chart.

In [ ]:
corpus_df = data["corpus_summary"].copy()
poster_results_df = data["poster_results"].copy()

poster_item_column = first_existing_column(
    poster_results_df,
    ["Poster item", "Item"],
    required=True,
)

poster_value_column = first_existing_column(
    poster_results_df,
    ["Value", "value"],
    required=True,
)

poster_lookup = {
    str(row[poster_item_column]): str(
        row[poster_value_column]
    )
    for _, row in poster_results_df.iterrows()
}

cards = [
    (
        "Reviewed notes",
        poster_lookup.get(
            "Reviewed notes",
            "70",
        ),
        NAVY,
    ),
    (
        "Gold mentions",
        poster_lookup.get(
            "Gold mentions",
            "491",
        ),
        BLUE,
    ),
    (
        "Predictions",
        poster_lookup.get(
            "System predictions",
            "546",
        ),
        PURPLE,
    ),
    (
        "Exact-span F1",
        poster_lookup.get(
            "Exact-span F1",
            f"{pipeline_values[0]:.3f}",
        ),
        TEAL,
    ),
    (
        "Strict full F1",
        poster_lookup.get(
            "Strict full extraction F1",
            f"{pipeline_values[-1]:.3f}",
        ),
        AMBER,
    ),
]

figure, axis = plt.subplots(figsize=(14, 3.2))
axis.set_xlim(0, len(cards))
axis.set_ylim(0, 1)
axis.axis("off")

for index, (
    title,
    value,
    colour,
) in enumerate(cards):
    rectangle = patches.FancyBboxPatch(
        (index + 0.08, 0.12),
        0.84,
        0.72,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        linewidth=0,
        facecolor=colour,
    )

    axis.add_patch(rectangle)

    axis.text(
        index + 0.50,
        0.58,
        value,
        ha="center",
        va="center",
        fontsize=25,
        fontweight="bold",
        color=WHITE,
    )

    axis.text(
        index + 0.50,
        0.29,
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        color=WHITE,
    )

axis.set_title(
    "Evaluation corpus and headline results",
    pad=10,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_03_poster_metric_cards",
    [
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
        SOCIAL_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

# Part B — Span-level error analysis

## Step 10 — Publication Figure 4: False-positive and false-negative category profile

In [ ]:
fp_df = data["fp_by_category"].copy()
fn_df = data["fn_by_category"].copy()

fp_category_column = first_existing_column(
    fp_df,
    ["category", "Category"],
    required=True,
)

fp_count_column = first_existing_column(
    fp_df,
    ["Count", "count"],
    required=True,
)

fn_category_column = first_existing_column(
    fn_df,
    ["category", "Category"],
    required=True,
)

fn_count_column = first_existing_column(
    fn_df,
    ["Count", "count"],
    required=True,
)

all_categories = sorted(
    set(fp_df[fp_category_column].astype(str))
    | set(fn_df[fn_category_column].astype(str))
)

error_profile_df = pd.DataFrame({
    "Category": all_categories,
})

error_profile_df = error_profile_df.merge(
    fp_df[
        [fp_category_column, fp_count_column]
    ].rename(
        columns={
            fp_category_column: "Category",
            fp_count_column: "False positives",
        }
    ),
    on="Category",
    how="left",
)

error_profile_df = error_profile_df.merge(
    fn_df[
        [fn_category_column, fn_count_column]
    ].rename(
        columns={
            fn_category_column: "Category",
            fn_count_column: "False negatives",
        }
    ),
    on="Category",
    how="left",
)

error_profile_df = error_profile_df.fillna(0)

error_profile_df["Total span errors"] = (
    error_profile_df["False positives"]
    + error_profile_df["False negatives"]
)

error_profile_df = (
    error_profile_df
    .sort_values(
        "Total span errors",
        ascending=False,
    )
    .head(12)
    .sort_values(
        "Total span errors",
        ascending=True,
    )
)

save_table(
    error_profile_df.sort_values(
        "Total span errors",
        ascending=False,
    ),
    "table_01_top_span_error_categories.csv",
)

figure, axis = plt.subplots(figsize=(10.5, 7))

y_positions = np.arange(len(error_profile_df))

axis.barh(
    y_positions,
    error_profile_df["False positives"],
    label="False positives",
    color=RED,
    alpha=0.88,
)

axis.barh(
    y_positions,
    error_profile_df["False negatives"],
    left=error_profile_df["False positives"],
    label="False negatives",
    color=AMBER,
    alpha=0.95,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    [
        clean_label(value)
        for value in error_profile_df["Category"]
    ]
)

axis.set_xlabel("Span-level error count")
axis.set_ylabel("Complication category")
axis.set_title(
    "Complication categories contributing most "
    "span-level errors"
)

axis.legend(
    ncol=2,
    loc="lower right",
)

axis.grid(
    axis="x",
    alpha=0.18,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_04_span_error_profile_by_category",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

## Step 11 — Publication Figure 5: False-negative root causes

In [ ]:
root_cause_df = data["fn_root_causes"].copy()

root_label_column = first_existing_column(
    root_cause_df,
    ["Cause", "cause", "root_cause"],
    required=True,
)

root_count_column = first_existing_column(
    root_cause_df,
    ["Count", "count"],
    required=True,
)

root_cause_df[root_label_column] = (
    root_cause_df[root_label_column]
    .map(clean_label)
)

root_cause_df = root_cause_df.sort_values(
    root_count_column,
    ascending=True,
)

figure, axis = plt.subplots(figsize=(9, 4.8))

bars = axis.barh(
    root_cause_df[root_label_column],
    root_cause_df[root_count_column],
    color=[TEAL, AMBER][:len(root_cause_df)],
)

axis.set_xlabel("False-negative count")
axis.set_ylabel("Root-cause classification")
axis.set_title(
    "Most missed mentions were note-level expression gaps, "
    "not unseen categories"
)

axis.bar_label(
    bars,
    padding=4,
    fontsize=10,
    fontweight="bold",
)

axis.grid(
    axis="x",
    alpha=0.18,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_05_false_negative_root_causes",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

# Part C — Category-level performance

## Step 12 — Publication Figure 6: Category F1 with gold-standard support

In [ ]:
category_df = data["category_metrics"].copy()

category_column = first_existing_column(
    category_df,
    ["Category", "category"],
    required=True,
)

category_f1_column = first_existing_column(
    category_df,
    ["F1-score", "F1", "f1"],
    required=True,
)

support_column = first_existing_column(
    category_df,
    [
        "Gold Mentions",
        "Gold mentions",
        "support",
        "Support",
    ],
    required=True,
)

category_plot_df = category_df[
    [
        category_column,
        category_f1_column,
        support_column,
    ]
].copy()

category_plot_df[category_f1_column] = pd.to_numeric(
    category_plot_df[category_f1_column],
    errors="coerce",
)

category_plot_df[support_column] = pd.to_numeric(
    category_plot_df[support_column],
    errors="coerce",
)

category_plot_df = (
    category_plot_df
    .dropna()
    .sort_values(
        [
            category_f1_column,
            support_column,
        ],
        ascending=[True, True],
    )
)

figure_height = max(
    7.0,
    0.38 * len(category_plot_df) + 1.8,
)

figure, axis = plt.subplots(
    figsize=(11, figure_height)
)

y_positions = np.arange(len(category_plot_df))

axis.hlines(
    y=y_positions,
    xmin=0,
    xmax=category_plot_df[category_f1_column],
    color=LIGHT_GREY,
    linewidth=3,
)

marker_sizes = (
    45
    + 7
    * np.sqrt(
        category_plot_df[support_column]
    )
)

scatter = axis.scatter(
    category_plot_df[category_f1_column],
    y_positions,
    s=marker_sizes,
    c=category_plot_df[support_column],
    cmap=LinearSegmentedColormap.from_list(
        "support_map",
        [AMBER, TEAL, NAVY],
    ),
    edgecolor=WHITE,
    linewidth=0.9,
    zorder=3,
)

axis.set_yticks(y_positions)
axis.set_yticklabels(
    [
        clean_label(value)
        for value
        in category_plot_df[category_column]
    ]
)

axis.set_xlim(0, 1.05)
axis.set_xlabel(
    "Exact-span + category F1-score"
)
axis.set_ylabel("Complication category")
axis.set_title(
    "Per-category performance with gold-standard support"
)

axis.axvline(
    float(
        performance_map[
            "Exact span + category"
        ][f1_column]
    ),
    linestyle="--",
    linewidth=1.4,
    color=RED,
    alpha=0.85,
    label="Overall F1",
)

axis.legend(loc="lower right")
axis.grid(
    axis="x",
    alpha=0.16,
)

colourbar = figure.colorbar(
    scatter,
    ax=axis,
    fraction=0.028,
    pad=0.02,
)
colourbar.set_label(
    "Gold-standard mentions"
)

figure.text(
    0.01,
    0.01,
    "Marker size and colour indicate support. "
    "Very high F1 for low-support categories should be interpreted cautiously.",
    fontsize=8.5,
    color=GREY,
)

figure.tight_layout(rect=[0, 0.035, 1, 1])

save_publication_figure(
    figure,
    "figure_06_category_f1_with_support",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

## Step 13 — Publication Figure 7: Focused category-error statement

The full 23-class confusion matrix is too dense for a poster. This compact figure communicates the clinically relevant category-error pattern directly.

In [ ]:
category_summary_df = data["category_summary"].copy()

summary_metric_column = first_existing_column(
    category_summary_df,
    ["Metric", "metric"],
)

summary_value_column = first_existing_column(
    category_summary_df,
    ["Value", "value"],
)

category_accuracy = np.nan

if (
    summary_metric_column is not None
    and summary_value_column is not None
):
    summary_lookup = {
        str(row[summary_metric_column]).strip().lower():
            row[summary_value_column]
        for _, row
        in category_summary_df.iterrows()
    }

    for key, value in summary_lookup.items():
        if "accuracy" in key:
            category_accuracy = float(value)
            break

if not np.isfinite(category_accuracy):
    category_accuracy = 0.9915

figure, axis = plt.subplots(figsize=(10, 4.8))
axis.set_xlim(0, 10)
axis.set_ylim(0, 4.8)
axis.axis("off")

left_box = patches.FancyBboxPatch(
    (0.45, 1.45),
    3.25,
    1.75,
    boxstyle="round,pad=0.04,rounding_size=0.08",
    facecolor="#FCE8E6",
    edgecolor=RED,
    linewidth=1.4,
)

right_box = patches.FancyBboxPatch(
    (6.30, 1.45),
    3.25,
    1.75,
    boxstyle="round,pad=0.04,rounding_size=0.08",
    facecolor="#E6F4F1",
    edgecolor=TEAL,
    linewidth=1.4,
)

axis.add_patch(left_box)
axis.add_patch(right_box)

axis.text(
    2.08,
    2.65,
    "Pneumonia",
    ha="center",
    va="center",
    fontsize=18,
    fontweight="bold",
    color=RED,
)

axis.text(
    2.08,
    1.95,
    "Gold-standard category",
    ha="center",
    va="center",
    fontsize=10,
    color=GREY,
)

axis.text(
    7.93,
    2.65,
    "Aspiration",
    ha="center",
    va="center",
    fontsize=18,
    fontweight="bold",
    color=TEAL,
)

axis.text(
    7.93,
    1.95,
    "Predicted category",
    ha="center",
    va="center",
    fontsize=10,
    color=GREY,
)

axis.annotate(
    "",
    xy=(6.15, 2.33),
    xytext=(3.85, 2.33),
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2.6,
        "color": NAVY,
    },
)

axis.text(
    5.00,
    2.70,
    "4 errors",
    ha="center",
    va="center",
    fontsize=13,
    fontweight="bold",
    color=NAVY,
)

axis.text(
    5.00,
    0.65,
    f"Exact-span category accuracy: "
    f"{category_accuracy:.2%}",
    ha="center",
    va="center",
    fontsize=16,
    fontweight="bold",
    color=NAVY,
)

axis.set_title(
    "All observed exact-span category errors followed "
    "one confusion pattern",
    pad=12,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_07_focused_category_error_pattern",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

# Part D — Contextual attributes

## Step 14 — Publication Figure 8: Assertion error matrix

The matrix is reconstructed from the error-pair table and displayed as row-normalised proportions. Correct classifications are stated separately in the accompanying summary.

In [ ]:
assertion_pairs_df = data[
    "assertion_error_pairs"
].copy()

assertion_gold_column = first_existing_column(
    assertion_pairs_df,
    ["gold_label", "Gold", "gold"],
    required=True,
)

assertion_pred_column = first_existing_column(
    assertion_pairs_df,
    [
        "predicted_label",
        "Predicted",
        "prediction",
    ],
    required=True,
)

assertion_count_column = first_existing_column(
    assertion_pairs_df,
    ["Count", "count"],
    required=True,
)

assertion_error_matrix = normalise_confusion_from_pairs(
    assertion_pairs_df,
    assertion_gold_column,
    assertion_pred_column,
    assertion_count_column,
)

plot_heatmap(
    assertion_error_matrix,
    "Assertion misclassification profile",
    "figure_08_assertion_error_profile",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

## Step 15 — Publication Figure 9: Temporality error matrix

In [ ]:
temporality_pairs_df = data[
    "temporality_error_pairs"
].copy()

temporality_gold_column = first_existing_column(
    temporality_pairs_df,
    ["gold_label", "Gold", "gold"],
    required=True,
)

temporality_pred_column = first_existing_column(
    temporality_pairs_df,
    [
        "predicted_label",
        "Predicted",
        "prediction",
    ],
    required=True,
)

temporality_count_column = first_existing_column(
    temporality_pairs_df,
    ["Count", "count"],
    required=True,
)

temporality_error_matrix = normalise_confusion_from_pairs(
    temporality_pairs_df,
    temporality_gold_column,
    temporality_pred_column,
    temporality_count_column,
)

plot_heatmap(
    temporality_error_matrix,
    "Temporality misclassification profile",
    "figure_09_temporality_error_profile",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

## Step 16 — Publication Figure 10: Contextual-attribute summary

Both conventional weighted F1 and support-aware macro F1 are shown so that class imbalance is transparent.

In [ ]:
def extract_summary_metric(
    dataframe,
    metric_name_fragment,
):
    metric_column = first_existing_column(
        dataframe,
        ["Metric", "metric"],
        required=True,
    )

    value_column = first_existing_column(
        dataframe,
        ["Value", "value"],
        required=True,
    )

    matches = dataframe.loc[
        dataframe[metric_column]
        .astype(str)
        .str.lower()
        .str.contains(
            metric_name_fragment.lower(),
            regex=False,
        )
    ]

    if matches.empty:
        return np.nan

    return float(matches.iloc[0][value_column])


assertion_research_df = data[
    "assertion_summary"
].copy()

temporality_research_df = data[
    "temporality_summary"
].copy()

context_metric_df = pd.DataFrame(
    [
        {
            "Attribute": "Assertion",
            "Accuracy": extract_summary_metric(
                assertion_research_df,
                "Accuracy",
            ),
            "Support-aware macro F1":
                extract_summary_metric(
                    assertion_research_df,
                    "Support-aware macro F1",
                ),
            "Weighted F1":
                extract_summary_metric(
                    assertion_research_df,
                    "Weighted F1",
                ),
        },
        {
            "Attribute": "Temporality",
            "Accuracy": extract_summary_metric(
                temporality_research_df,
                "Accuracy",
            ),
            "Support-aware macro F1":
                extract_summary_metric(
                    temporality_research_df,
                    "Support-aware macro F1",
                ),
            "Weighted F1":
                extract_summary_metric(
                    temporality_research_df,
                    "Weighted F1",
                ),
        },
    ]
)

save_table(
    context_metric_df.round(4),
    "table_02_context_attribute_metrics.csv",
)

figure, axis = plt.subplots(figsize=(9.5, 5.8))

x_positions = np.arange(
    len(context_metric_df)
)

metric_columns = [
    ("Accuracy", BLUE),
    ("Support-aware macro F1", AMBER),
    ("Weighted F1", TEAL),
]

bar_width = 0.24

for index, (
    metric_name,
    colour,
) in enumerate(metric_columns):
    offset = (
        index - 1
    ) * bar_width

    bars = axis.bar(
        x_positions + offset,
        context_metric_df[metric_name],
        width=bar_width,
        label=metric_name,
        color=colour,
    )

    axis.bar_label(
        bars,
        labels=[
            f"{value:.3f}"
            for value
            in context_metric_df[metric_name]
        ],
        padding=3,
        fontsize=8.5,
    )

axis.set_xticks(x_positions)
axis.set_xticklabels(
    context_metric_df["Attribute"]
)

axis.set_ylim(0, 1.05)
axis.set_ylabel("Score")
axis.set_title(
    "Contextual attribute performance is sensitive "
    "to class imbalance"
)

axis.legend(
    ncol=3,
    loc="upper right",
)

axis.grid(
    axis="y",
    alpha=0.18,
)

figure.text(
    0.01,
    0.01,
    "Support-aware macro F1 averages only classes represented "
    "in the gold standard; weighted F1 remains dominated by common classes.",
    fontsize=8.5,
    color=GREY,
)

figure.tight_layout(rect=[0, 0.05, 1, 1])

save_publication_figure(
    figure,
    "figure_10_context_attribute_summary",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

# Part E — Integrated error interpretation

## Step 17 — Publication Figure 11: Primary error taxonomy

This figure uses mutually exclusive primary error categories and is therefore safe for poster use.

In [ ]:
taxonomy_df = data[
    "primary_error_taxonomy"
].copy()

taxonomy_label_column = first_existing_column(
    taxonomy_df,
    [
        "Primary error type",
        "Error type",
        "Category",
    ],
    required=True,
)

taxonomy_count_column = first_existing_column(
    taxonomy_df,
    ["Count", "count"],
    required=True,
)

taxonomy_df = taxonomy_df.sort_values(
    taxonomy_count_column,
    ascending=True,
)

figure, axis = plt.subplots(figsize=(10, 6))

bars = axis.barh(
    taxonomy_df[taxonomy_label_column]
    .map(clean_label),
    taxonomy_df[taxonomy_count_column],
    color=[
        RED
        if "false positive" in str(label).lower()
        else AMBER
        if "false negative" in str(label).lower()
        else TEAL
        if "category" in str(label).lower()
        else PURPLE
        for label in taxonomy_df[
            taxonomy_label_column
        ]
    ],
)

axis.set_xlabel("Mutually exclusive error count")
axis.set_ylabel("Primary error type")
axis.set_title(
    "Primary error taxonomy for the evaluated NLP system"
)

axis.bar_label(
    bars,
    padding=4,
    fontsize=9,
)

axis.grid(
    axis="x",
    alpha=0.18,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_11_primary_error_taxonomy",
    [
        DISSERTATION_DIRECTORY,
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

## Step 18 — Publication Figure 12: Key findings panel

This infographic is suitable for the poster conclusion area and presentation summary slide.

In [ ]:
headline_findings = [
    (
        "Strong mention detection",
        "Exact-span F1 = 0.903",
        TEAL,
    ),
    (
        "Reliable category assignment",
        "464 of 468 exact spans correct",
        BLUE,
    ),
    (
        "Partial boundary disagreement",
        "Relaxed category F1 = 0.924",
        PURPLE,
    ),
    (
        "Main weakness",
        "Assertion and temporality reduce full F1 to 0.577",
        AMBER,
    ),
]

figure, axis = plt.subplots(figsize=(12, 6.3))
axis.set_xlim(0, 2)
axis.set_ylim(0, 2)
axis.axis("off")

positions = [
    (0.08, 1.08),
    (1.04, 1.08),
    (0.08, 0.08),
    (1.04, 0.08),
]

for (
    title,
    detail,
    colour,
), (
    x_position,
    y_position,
) in zip(
    headline_findings,
    positions,
):
    rectangle = patches.FancyBboxPatch(
        (x_position, y_position),
        0.88,
        0.78,
        boxstyle="round,pad=0.03,rounding_size=0.04",
        facecolor=WHITE,
        edgecolor=colour,
        linewidth=2.0,
    )

    axis.add_patch(rectangle)

    axis.add_patch(
        patches.Rectangle(
            (x_position, y_position),
            0.08,
            0.78,
            facecolor=colour,
            edgecolor="none",
        )
    )

    axis.text(
        x_position + 0.14,
        y_position + 0.53,
        title,
        ha="left",
        va="center",
        fontsize=13,
        fontweight="bold",
        color=NAVY,
    )

    axis.text(
        x_position + 0.14,
        y_position + 0.25,
        detail,
        ha="left",
        va="center",
        fontsize=10.5,
        color=GREY,
        wrap=True,
    )

axis.set_title(
    "Key findings from the clinical NLP evaluation",
    fontsize=17,
    pad=14,
)

figure.tight_layout()

save_publication_figure(
    figure,
    "figure_12_key_findings_panel",
    [
        POSTER_DIRECTORY,
        PRESENTATION_DIRECTORY,
        SOCIAL_DIRECTORY,
    ],
)

plt.show()
plt.close(figure)

# Part F — Final asset packages

## Step 19 — Create dissertation, poster and presentation manifests

In [ ]:
asset_manifest_df = pd.DataFrame(
    [
        {
            "Figure": "figure_01_evaluation_performance_ladder",
            "Dissertation section": "Results — Overall performance",
            "Poster use": "Primary results figure",
            "Presentation use": "Results slide",
        },
        {
            "Figure": "figure_02_contextual_attribute_penalty",
            "Dissertation section": "Results — End-to-end performance",
            "Poster use": "Optional",
            "Presentation use": "Interpretation slide",
        },
        {
            "Figure": "figure_03_poster_metric_cards",
            "Dissertation section": "Not required",
            "Poster use": "Headline statistics",
            "Presentation use": "Study overview",
        },
        {
            "Figure": "figure_04_span_error_profile_by_category",
            "Dissertation section": "Error analysis",
            "Poster use": "Error-analysis panel",
            "Presentation use": "Error-analysis slide",
        },
        {
            "Figure": "figure_05_false_negative_root_causes",
            "Dissertation section": "Error analysis",
            "Poster use": "Optional",
            "Presentation use": "Error-analysis slide",
        },
        {
            "Figure": "figure_06_category_f1_with_support",
            "Dissertation section": "Per-category results",
            "Poster use": "Per-category panel",
            "Presentation use": "Category-results slide",
        },
        {
            "Figure": "figure_07_focused_category_error_pattern",
            "Dissertation section": "Category error analysis",
            "Poster use": "Compact category-error statement",
            "Presentation use": "Category-error slide",
        },
        {
            "Figure": "figure_08_assertion_error_profile",
            "Dissertation section": "Contextual attributes",
            "Poster use": "Optional",
            "Presentation use": "Assertion slide",
        },
        {
            "Figure": "figure_09_temporality_error_profile",
            "Dissertation section": "Contextual attributes",
            "Poster use": "Optional",
            "Presentation use": "Temporality slide",
        },
        {
            "Figure": "figure_10_context_attribute_summary",
            "Dissertation section": "Contextual attributes",
            "Poster use": "Context panel",
            "Presentation use": "Context slide",
        },
        {
            "Figure": "figure_11_primary_error_taxonomy",
            "Dissertation section": "Error taxonomy",
            "Poster use": "Primary error panel",
            "Presentation use": "Error summary",
        },
        {
            "Figure": "figure_12_key_findings_panel",
            "Dissertation section": "Not required",
            "Poster use": "Conclusion panel",
            "Presentation use": "Conclusion slide",
        },
    ]
)

display(asset_manifest_df)

save_table(
    asset_manifest_df,
    "table_03_final_asset_manifest.csv",
)

## Step 20 — Export concise dissertation results table

In [ ]:
dissertation_results_table_df = (
    performance_df[
        [
            evaluation_column,
            precision_column,
            recall_column,
            f1_column,
        ]
    ]
    .rename(
        columns={
            evaluation_column:
                "Evaluation level",

            precision_column:
                "Precision",

            recall_column:
                "Recall",

            f1_column:
                "F1-score",
        }
    )
    .copy()
)

for metric_column in [
    "Precision",
    "Recall",
    "F1-score",
]:
    dissertation_results_table_df[
        metric_column
    ] = (
        dissertation_results_table_df[
            metric_column
        ]
        .astype(float)
        .round(4)
    )

display(dissertation_results_table_df)

save_table(
    dissertation_results_table_df,
    "table_04_dissertation_overall_results.csv",
)

## Step 21 — Final validation

Notebook 09 validates:

- the frozen Notebook 08 input;
- all 12 publication figures;
- all expected file formats;
- final table exports;
- clean, non-obsolete filenames.

In [ ]:
expected_figure_stems = [
    "figure_01_evaluation_performance_ladder",
    "figure_02_contextual_attribute_penalty",
    "figure_03_poster_metric_cards",
    "figure_04_span_error_profile_by_category",
    "figure_05_false_negative_root_causes",
    "figure_06_category_f1_with_support",
    "figure_07_focused_category_error_pattern",
    "figure_08_assertion_error_profile",
    "figure_09_temporality_error_profile",
    "figure_10_context_attribute_summary",
    "figure_11_primary_error_taxonomy",
    "figure_12_key_findings_panel",
]

expected_destinations = {
    "dissertation": DISSERTATION_DIRECTORY,
    "poster": POSTER_DIRECTORY,
    "presentation": PRESENTATION_DIRECTORY,
}

validation_rows = []

for figure_stem in expected_figure_stems:
    destination_presence = {}

    for destination_name, destination_path in (
        expected_destinations.items()
    ):
        destination_presence[
            destination_name
        ] = all(
            (
                destination_path
                / f"{figure_stem}.{extension}"
            ).exists()
            for extension
            in ["png", "pdf", "svg"]
        )

    validation_rows.append(
        {
            "Check":
                f"{figure_stem}_generated",

            "Passed":
                any(
                    destination_presence.values()
                ),

            "Details":
                ", ".join(
                    f"{name}={value}"
                    for name, value
                    in destination_presence.items()
                ),
        }
    )

required_tables = [
    TABLE_DIRECTORY
    / "table_01_top_span_error_categories.csv",

    TABLE_DIRECTORY
    / "table_02_context_attribute_metrics.csv",

    TABLE_DIRECTORY
    / "table_03_final_asset_manifest.csv",

    TABLE_DIRECTORY
    / "table_04_dissertation_overall_results.csv",
]

validation_rows.extend(
    [
        {
            "Check":
                "all_required_tables_generated",

            "Passed":
                all(
                    path.exists()
                    for path in required_tables
                ),

            "Details":
                f"{sum(path.exists() for path in required_tables)}"
                f"/{len(required_tables)} tables",
        },
        {
            "Check":
                "no_obsolete_figure_14_name_created",

            "Passed":
                not any(
                    OUTPUT_DIRECTORY.rglob(
                        "figure_14_per_category_f1.*"
                    )
                ),

            "Details":
                "Legacy Notebook 08 filename excluded",
        },
        {
            "Check":
                "notebook_08_validation_preserved",

            "Passed":
                failed_validation_df.empty,

            "Details":
                "Notebook 08 frozen validation passed",
        },
    ]
)

notebook_09_validation_df = pd.DataFrame(
    validation_rows
)

display(notebook_09_validation_df)

validation_output_path = (
    REPORT_DIRECTORY
    / "notebook_09_final_validation.csv"
)

notebook_09_validation_df.to_csv(
    validation_output_path,
    index=False,
)

failed_notebook_09_checks = (
    notebook_09_validation_df.loc[
        ~notebook_09_validation_df[
            "Passed"
        ]
    ]
)

if not failed_notebook_09_checks.empty:
    display(failed_notebook_09_checks)

    raise RuntimeError(
        "One or more Notebook 09 validation checks failed."
    )

print(
    "\nAll Notebook 09 validation checks passed."
)

## Step 22 — Save final Notebook 09 report

In [ ]:
notebook_09_report = {
    "notebook":
        "09",

    "title":
        "Publication-Quality Visualisations "
        "and Final Research Assets",

    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "input_directory":
        str(NOTEBOOK_08_DIRECTORY),

    "output_directory":
        str(OUTPUT_DIRECTORY),

    "figures_generated":
        expected_figure_stems,

    "figure_count":
        len(expected_figure_stems),

    "tables_generated":
        [
            path.name
            for path
            in sorted(
                TABLE_DIRECTORY.glob("*.csv")
            )
        ],

    "notebook_08_validation_passed":
        bool(failed_validation_df.empty),

    "notebook_09_validation_passed":
        bool(
            failed_notebook_09_checks.empty
        ),

    "research_integrity":
        {
            "evaluation_recomputed":
                False,

            "gold_standard_modified":
                False,

            "predictions_modified":
                False,

            "patient_text_in_public_figures":
                False,

            "figures_derived_from_frozen_notebook_08_outputs":
                True,
        },
}

report_path = (
    REPORT_DIRECTORY
    / "notebook_09_publication_assets_report.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook_09_report,
        file,
        indent=2,
    )

print(f"[SAVE] {report_path}")
display(notebook_09_report)

# Notebook 09 completion criteria

Notebook 09 is complete when:

1. Notebook 08 validation passes.
2. All 12 final figure groups are generated.
3. Each applicable figure is exported as PNG, PDF and SVG.
4. The dissertation, poster and presentation manifests are saved.
5. No evaluation metric is recomputed or changed.
6. No raw patient text appears in public-facing figures.
7. `notebook_09_final_validation.csv` contains no failed checks.
8. `notebook_09_publication_assets_report.json` is created.

After this notebook passes, the analytical pipeline is frozen and the next work should be:

- final poster construction;
- dissertation Results chapter;
- dissertation Discussion chapter;
- presentation slides.